
用途：教学演示 - 完整的 RAG 流程闭环

核心流程：
  文档加载 → 文档切片 → Embedding 向量化 → 存入 Milvus → 检索 → RAG 问答
###### 

---
运行前检查
---
1. 已安装依赖：pip install -r ../requirements.txt
2. 已配置 Milvus（使用远程服务）
3. 配置 API Key：确保 .env 文件中配置了 ALIYUN_API_KEY 或 DASHSCOPE_API_KEY
---

从项目根目录（wolin_learn-master/）运行本文件：
  python rag_examples/05_rag_pipeline/rag_full_pipeline.py

加载 .env 文件中的环境变量

In [ ]:
from dotenv import load_dotenv


def demo_full_pipeline():
    """
    演示完整 RAG 流程
    """
    print("=" * 70)
    print("         RAG 完整流程演示")
    print("=" * 70)

    # 初始化 Pipeline（使用阿里云 Embedding API）
    pipeline = RAGPipeline(
        milvus_uri=MILVUS_URI,  # 已在文件顶部从 milvus_config 导入
        collection_name="demo_rag",
        dim=1024  # text-embedding-v4 维度
    )

    # 准备测试文件（使用绝对路径）
    base_dir = os.path.dirname(os.path.abspath(__file__))
    data_dir = os.path.join(base_dir, '..', 'data', 'txt')
    file_paths = [
        os.path.join(data_dir, "milvus_intro.txt"),
        os.path.join(data_dir, "ai_intro.txt"),
    ]

    # 运行完整流程
    pipeline.run_full_pipeline(
        file_paths=file_paths,
        chunk_size=300,
        chunk_overlap=50,
        chunk_method="sliding_window",
        index_type="FLAT"
    )

    # 测试问答
    print("\n" + "-" * 50)
    print("测试问答：")
    print("-" * 50)

    questions = [
        "Milvus 是什么？",
        "人工智能包括哪些技术？"
    ]

    for question in questions:
        print(f"\n🙋 用户：{question}")
        result = pipeline.ask(question, top_k=3)
        print(f"🤖 助手：{result['answer']}")

        if result['sources']:
            print("\n📚 引用来源:")
            for i, src in enumerate(result['sources'][:2], 1):
                print(f"  [{i}] {src['content'][:50]}...")

    pipeline.close()


# =============================================================================
# 流程图总结
# =============================================================================

In [ ]:
from dotenv import load_dotenv


def pipeline_summary():
    """
    完整流程总结
    """
    print()
    print("=" * 60)
    print("RAG 完整流程总结")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 完整 RAG 流程（可复用模板）                               │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  # 1. 初始化 Pipeline                                    │
│  from milvus_config import MILVUS_URI                   │
│  pipeline = RAGPipeline(                                │
│      milvus_uri=MILVUS_URI,  # 使用远程 Milvus 服务      │
│      embedding_model="local",                           │
│      dim=1024                                           │
│  )                                                      │
│                                                         │
│  # 2. 运行完整流程（构建知识库）                         │
│  pipeline.run_full_pipeline(                            │
│      file_paths=["doc1.txt", "doc2.txt"],               │
│      chunk_size=500,                                    │
│      chunk_overlap=50                                   │
│  )                                                      │
│                                                         │
│  # 3. 问答                                              │
│  result = pipeline.ask("你的问题？", top_k=5)           │
│  print(result['answer'])                                │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 关键点：
1. Embedding 模型要一致（文档和查询用同一模型）
2. 向量维度要与模型匹配
3. 切片大小影响检索效果（建议 300-500）
4. 建表前先删表，确保可重复运行
""")


# =============================================================================
# 主程序入口
# =============================================================================